#Import dataset

In [6]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd

file_path = '/content/drive/My Drive/BT4222/df_feature_engineering.parquet'
df = pd.read_parquet(file_path)
df.head()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


,id,label,statement,subject,speaker,speaker_job_title,state_info,party_affiliation,barely_true_counts,false_counts,...,label_pants-fire,label_true,statement_embedding,subject_embedding,state_info_encoded,party_affiliation_encoded,speaker_job_title_embedding,context_embedding,sentiment_polarity,sentiment_subjectivity
0,7874.json,half-true,new jersey pension system solvent,"pensions,state-finances",13th district gop slate,unknown,new jersey,republican,0.0,0.0,...,0,0,"[-0.012083505280315876, 0.04291439801454544, -...","[-0.00025620736414566636, 0.07373839616775513,...",31,8,"[-0.043102607131004333, 0.06563703715801239, -...","[-0.09551206976175308, 0.009169962257146835, 0...",0.136364,0.455
1,2287.json,pants-fire,president obama muslim,"candidates-biography,obama-birth-certificate,r...",18 percent american public,unknown,unknown,other,0.0,0.0,...,1,0,"[0.038615476340055466, 0.09793295711278915, -0...","[0.018359245732426643, -0.0027450439520180225,...",45,7,"[-0.043102607131004333, 0.06563703715801239, -...","[-0.024317895993590355, -0.01789085566997528, ...",0.000000,0.000
2,13412.json,true,test share fact widget,technology,18 percent american public,unknown,unknown,other,0.0,0.0,...,0,1,"[0.008107000961899757, 0.016036679968237877, -...","[-0.053375788033008575, 0.08707486093044281, -...",45,7,"[-0.043102607131004333, 0.06563703715801239, -...","[-0.06791266053915024, -0.015540316700935364, ...",0.000000,0.000
3,2426.json,barely-true,new health care law cut billion medicare hurt ...,"health-care,medicare,message-machine",60 plus association,unknown,unknown,other,2.0,0.0,...,0,0,"[-0.05084792524576187, 0.031769607216119766, 0...","[-0.01470591127872467, 0.015529205091297626, 0...",45,7,"[-0.043102607131004333, 0.06563703715801239, -...","[-0.10175436735153198, -0.002838208107277751, ...",0.136364,0.455
4,4832.json,pants-fire,independent payment advisory board created hea...,"government-regulation,health-care,message-mach...",60 plus association,unknown,unknown,other,2.0,0.0,...,1,0,"[-0.10781728476285934, 0.017993967980146408, 0...","[0.02528245933353901, 0.005831953138113022, -0...",45,7,"[-0.043102607131004333, 0.06563703715801239, -...","[-0.03631351888179779, 0.039651136845350266, -...",0.068182,0.290


In [5]:
df.columns.tolist()

['id',
 'label',
 'statement',
 'subject',
 'speaker',
 'speaker_job_title',
 'state_info',
 'party_affiliation',
 'barely_true_counts',
 'false_counts',
 'half_true_counts',
 'mostly_true_counts',
 'pants_on_fire_counts',
 'context',
 'date',
 'true_counts',
 'total_counts',
 'label_barely-true',
 'label_false',
 'label_half-true',
 'label_mostly-true',
 'label_pants-fire',
 'label_true',
 'statement_embedding',
 'subject_embedding',
 'state_info_encoded',
 'party_affiliation_encoded',
 'speaker_job_title_embedding',
 'context_embedding',
 'sentiment_polarity',
 'sentiment_subjectivity']

# Group by speaker to get counts


In [22]:
import pandas as pd
import numpy as np

df_new = df.copy()

# Make sure 'date' is datetime
df_new['date'] = pd.to_datetime(df_new['date'])

#for data inspection: (btw total count is calc wrongly LOL)
# Sort by speaker and date
df_new = df_new.sort_values(by=['speaker', 'date'])

df_new.head()

## Model 1: Leaving metadata features unaggregated across speakers

## Categorising into true/false statements to generate binary reliability classifier

In [27]:
# Classification of what is true/false
# trueish_labels = ['true', 'mostly-true', 'half-true']
# falseish_labels = ['barely-true', 'false', 'pants-fire']

df_1 = df_new.copy()
# initially the counts were weighted equally but that gives a problem of comparing trueish and falseish because if there is 1 count of pants of fire it wouldnt over ride 1 count of barely-true
# Assign weights
weights = {
    'true_counts': 0.75,
    'mostly_true_counts': 0.5,
    'half_true_counts': 0.25,
    'barely_true_counts': -0.25,
    'false_counts': -0.75,
    'pants_on_fire_counts': -1.0
}

# Apply weighted sum for a reliability score
df_1['weighted_score'] = (
    df_1['true_counts'] * weights['true_counts'] +
    df_1['mostly_true_counts'] * weights['mostly_true_counts'] +
    df_1['half_true_counts'] * weights['half_true_counts'] +
    df_1['barely_true_counts'] * weights['barely_true_counts'] +
    df_1['false_counts'] * weights['false_counts'] +
    df_1['pants_on_fire_counts'] * weights['pants_on_fire_counts']
)

# Set reliability: 1 if score > 0, else 0
df_1['reliability'] = (df_1['weighted_score'] > 0).astype(int)

df_1.head()

,id,label,statement,subject,speaker,speaker_job_title,state_info,party_affiliation,barely_true_counts,false_counts,...,statement_embedding,subject_embedding,state_info_encoded,party_affiliation_encoded,speaker_job_title_embedding,context_embedding,sentiment_polarity,sentiment_subjectivity,weighted_score,reliability
0,7874.json,half-true,new jersey pension system solvent,"pensions,state-finances",13th district gop slate,unknown,new jersey,republican,0.0,0.0,...,"[-0.012083505280315876, 0.04291439801454544, -...","[-0.00025620736414566636, 0.07373839616775513,...",31,8,"[-0.043102607131004333, 0.06563703715801239, -...","[-0.09551206976175308, 0.009169962257146835, 0...",0.136364,0.455,0.25,1
2,13412.json,true,test share fact widget,technology,18 percent american public,unknown,unknown,other,0.0,0.0,...,"[0.008107000961899757, 0.016036679968237877, -...","[-0.053375788033008575, 0.08707486093044281, -...",45,7,"[-0.043102607131004333, 0.06563703715801239, -...","[-0.06791266053915024, -0.015540316700935364, ...",0.000000,0.000,-0.25,0
1,2287.json,pants-fire,president obama muslim,"candidates-biography,obama-birth-certificate,r...",18 percent american public,unknown,unknown,other,0.0,0.0,...,"[0.038615476340055466, 0.09793295711278915, -0...","[0.018359245732426643, -0.0027450439520180225,...",45,7,"[-0.043102607131004333, 0.06563703715801239, -...","[-0.024317895993590355, -0.01789085566997528, ...",0.000000,0.000,-0.25,0
3,2426.json,barely-true,new health care law cut billion medicare hurt ...,"health-care,medicare,message-machine",60 plus association,unknown,unknown,other,2.0,0.0,...,"[-0.05084792524576187, 0.031769607216119766, 0...","[-0.01470591127872467, 0.015529205091297626, 0...",45,7,"[-0.043102607131004333, 0.06563703715801239, -...","[-0.10175436735153198, -0.002838208107277751, ...",0.136364,0.455,-1.50,0
4,4832.json,pants-fire,independent payment advisory board created hea...,"government-regulation,health-care,message-mach...",60 plus association,unknown,unknown,other,2.0,0.0,...,"[-0.10781728476285934, 0.017993967980146408, 0...","[0.02528245933353901, 0.005831953138113022, -0...",45,7,"[-0.043102607131004333, 0.06563703715801239, -...","[-0.03631351888179779, 0.039651136845350266, -...",0.068182,0.290,-1.50,0


# Filter dataframe to only include metadata features we are using


In [32]:
# Select features
embedding_cols = [
    'statement_embedding',
    'subject_embedding',
    'speaker_job_title_embedding',
    'context_embedding'
]

flat_embeddings = np.hstack([
    np.vstack(df_1[col].values) for col in embedding_cols
])

# Select non-embedding features
meta_cols = [
    'state_info_encoded',
    'party_affiliation_encoded',
    'sentiment_polarity',
    'sentiment_subjectivity'
]
meta_features = df_1[meta_cols].values

# Combine all features
X = np.hstack([flat_embeddings, meta_features])

# Target label (reliability)
y = df_1['reliability'].values

# Split train/test/val set

In [33]:
from sklearn.model_selection import train_test_split

# First split: train + temp (which will be split again into val + test)
X_train_full, X_test, y_train_full, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Second split: from training set, take 20% as validation
X_train, X_val, y_train, y_val = train_test_split(
    X_train_full, y_train_full, test_size=0.2, random_state=42, stratify=y_train_full
)

#Convert into PyTorch TensorDatasets and wrap them in DataLoaders

In [50]:
import torch
from torch.utils.data import TensorDataset, DataLoader

# Convert to tensors (ensure float32 for features, long/int for labels)
X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.float32)

X_val_tensor = torch.tensor(X_val, dtype=torch.float32)
y_val_tensor = torch.tensor(y_val, dtype=torch.float32)

X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test, dtype=torch.float32)

# Wrap in TensorDataset
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
val_dataset = TensorDataset(X_val_tensor, y_val_tensor)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)

# Create DataLoaders
batch_size = 128

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size)
test_loader = DataLoader(test_dataset, batch_size=batch_size)


#Define MLP Model

In [51]:
import torch.nn as nn
import torch.nn.functional as F

class MLPClassifier(nn.Module):
    def __init__(self, input_dim, hidden_dim=128, dropout_prob=0.3):
        super(MLPClassifier, self).__init__()

        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.bn1 = nn.BatchNorm1d(hidden_dim)
        self.dropout1 = nn.Dropout(p=dropout_prob)

        self.fc2 = nn.Linear(hidden_dim, hidden_dim // 2)
        self.bn2 = nn.BatchNorm1d(hidden_dim // 2)
        self.dropout2 = nn.Dropout(p=dropout_prob)

        self.out = nn.Linear(hidden_dim // 2, 1)

    def forward(self, x):
        x = F.relu(self.bn1(self.fc1(x)))
        x = self.dropout1(x)
        x = F.relu(self.bn2(self.fc2(x)))
        x = self.dropout2(x)
        return torch.sigmoid(self.out(x))


input_dim = X_train.shape[1]
model = MLPClassifier(input_dim=input_dim).to("cuda" if torch.cuda.is_available() else "cpu")

#Training Loop

In [53]:
from torch.optim import Adam

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
criterion = nn.BCELoss()
optimizer = Adam(model.parameters(), lr=0.0015)

epochs = 10

for epoch in range(epochs):
    model.train()
    total_loss = 0
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device).unsqueeze(1)

        preds = model(xb)
        loss = criterion(preds, yb)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    # Validation loop
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for xb, yb in val_loader:
            xb, yb = xb.to(device), yb.to(device)
            preds = model(xb).squeeze() > 0.5
            correct += (preds == yb).sum().item()
            total += yb.size(0)

    val_acc = (correct / total) * 100
    print(f"Epoch {epoch+1}/{epochs} - Loss: {total_loss:.4f} - Val Acc: {val_acc:.4f}%")


Epoch 1/10 - Loss: 25.3629 - Val Acc: 73.6070%
Epoch 2/10 - Loss: 23.0053 - Val Acc: 73.4115%
Epoch 3/10 - Loss: 21.8641 - Val Acc: 73.4115%
Epoch 4/10 - Loss: 20.8897 - Val Acc: 73.1672%
Epoch 5/10 - Loss: 19.4967 - Val Acc: 74.2913%
Epoch 6/10 - Loss: 18.3676 - Val Acc: 73.8514%
Epoch 7/10 - Loss: 17.4425 - Val Acc: 74.5846%
Epoch 8/10 - Loss: 16.6114 - Val Acc: 74.2913%
Epoch 9/10 - Loss: 15.4691 - Val Acc: 72.8250%
Epoch 10/10 - Loss: 14.5455 - Val Acc: 74.4868%
